In [11]:
# ── CELLULE 1 : Chargement des données ──
import pandas as pd
import os, csv
from sklearn.preprocessing import LabelEncoder

Proj = "PRJNA755688-stage12"
path = "/mnt/g/ngs/data/" + Proj
path1 = path + "/"
path2 = path + "/rna/"  # 
fichiers_non_trouve = []

f = path1 + "liste-files.csv"
df_tot = pd.DataFrame()

with open(f, 'r') as csvfile:
    filereader = csv.reader(csvfile, delimiter=',')
    for row in filereader:
        sample = row[0]
        label = row[1]

        f1 = path2 + "results-" + sample + ".txt"
        if os.path.isfile(f1):
            df = pd.read_csv(f1, sep="\t")
            df["sample"] = sample
            df_pivot = df.pivot_table(values=['count'], columns='RNA', index=['sample'])

            if "I" in label:
                label = "II"
            if "healthy" in label:
                label = "control"
            if "Normal" in label:
                label = "control"

            df_pivot["label"] = label
            df_tot = pd.concat([df_tot, df_pivot])
        else:
            fichiers_non_trouve.append(f1)

print("Fichiers non trouvés :", len(fichiers_non_trouve))
 #print(df_tot.head())

encoder = LabelEncoder()
print(df_tot.label.unique())

print(df_tot.label.value_counts())

target = encoder.fit_transform(df_tot.label)
df_tot = df_tot.fillna(0)

Fichiers non trouvés : 0
['II' 'control']
label
control    561
II         399
Name: count, dtype: int64


In [12]:
# ── CELLULE 2 : Split train/test + Validation croisée 5-fold ──
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix, classification_report
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.decomposition import PCA
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Split stratifié : 80% train, 20% test final (jamais utilisé dans la CV) ──
labels = df_tot["label"].copy()
idx_train, idx_test = train_test_split(
    df_tot.index, test_size=0.20, random_state=222, stratify=labels
)

df_train_all = df_tot.loc[idx_train].copy()
y_train_all = labels.loc[idx_train].copy()
df_test_final = df_tot.loc[idx_test].copy()
y_test_final = labels.loc[idx_test].copy()

print(f"Train set: {df_train_all.shape}  ({dict(y_train_all.value_counts())})")
print(f"Test set (réservé pour évaluation finale): {df_test_final.shape}  ({dict(y_test_final.value_counts())})")
print("[OK] Split effectué, le test est mis de côté jusqu'à la fin")

# ── Définition du pipeline de preprocessing + ML ──
def run_pipeline(train_df, train_labels, test_df=None, random_state=42):
    """
    Pipeline complet (doit être ré-exécuté dans chaque fold de CV) :
      1. Undersampling des classes majoritaires (sur train uniquement)
      2. Filtrage des RNAs : exprimés >20% des échantillons, puis top 50 par variance
      3. NCA + PCA (réduction dimensionnelle)
      4. RandomForest Classifier (avec régularisation)
    Retourne le modèle, les prédictions (sous forme de labels originaux), et les données transformées.
    """
    le_inner = LabelEncoder()
    
    # 1. Undersampling
    class_counts = train_labels.value_counts()
    min_count = class_counts.min()
    idx_balanced = []
    for cls in class_counts.index:
        cls_idx = train_labels[train_labels == cls].index
        if len(cls_idx) > min_count:
            cls_idx = cls_idx.to_series().sample(n=min_count, random_state=random_state)
        idx_balanced.extend(cls_idx.tolist())
    
    X_bal = train_df.loc[idx_balanced].copy()
    y_bal = train_labels.loc[idx_balanced].copy()
    y_fit = le_inner.fit_transform(y_bal)
    
    # 2. Filtrage des RNAs (sur le train uniquement)
    nonzero_pct = (X_bal > 0).sum(axis=0) / X_bal.shape[0]
    mask_exprime = nonzero_pct >= 0.20
    X_f1 = X_bal.loc[:, mask_exprime]
    if X_f1.shape[1] > 50:
        variances = X_f1.var(axis=0).sort_values(ascending=False)
        top_feats = variances.head(50).index
        X_f1 = X_f1[top_feats]
    # print(f"    RNAs retenus: {X_f1.shape[1]}")
    
    # 3. NCA + PCA
    nca = NeighborhoodComponentsAnalysis(random_state=random_state)
    X_nca = nca.fit_transform(X_f1, y_fit)
    pca = PCA(n_components=0.95, random_state=random_state)
    X_pca = pca.fit_transform(X_nca)
    print(f"    Composantes PCA: {X_pca.shape[1]}")
    
    # 4. RandomForest avec régularisation (limite la profondeur)
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=5,
        random_state=random_state, class_weight="balanced"
    )
    rf.fit(X_pca, y_fit)
    
    if test_df is not None:
        X_test_proc = test_df.copy()
        if "label" in X_test_proc.columns:
            X_test_proc = X_test_proc.drop(columns=["label"])
        X_test_f1 = X_test_proc.loc[:, X_test_proc.columns.intersection(X_f1.columns)]
        X_test_nca = nca.transform(X_test_f1)
        X_test_pca = pca.transform(X_test_nca)
        y_pred_int = rf.predict(X_test_pca)
        y_pred = le_inner.inverse_transform(y_pred_int)  # ← reconversion en labels originaux
        y_true = test_df["label"] if "label" in test_df.columns else None
        return rf, le_inner, y_pred, y_true
    else:
        return rf, le_inner, None, None

# ── Validation croisée Stratified 5-fold SUR LE TRAIN UNIQUEMENT ──
# Chaque fold refait undersampling + filtrage + NCA+PCA de zéro
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=222)

cv_scores_acc = []
cv_scores_bal = []
cv_scores_mcc = []
all_y_true = []
all_y_pred = []
fold = 1

print("\n" + "=" * 60)
print("VALIDATION CROISÉE 5-FOLD")
print("=" * 60)

df_cv = df_train_all.copy()
df_cv["label"] = y_train_all

for train_idx, val_idx in skf.split(df_cv, y_train_all):
    print(f"\n  ── Fold {fold} ──")
    df_fold_train = df_cv.iloc[train_idx]
    df_fold_val = df_cv.iloc[val_idx]
    
    y_fold_train = df_fold_train["label"]
    y_fold_val = df_fold_val["label"]
    
    # Pipeline complet sur ce fold
    _, _, y_pred, _ = run_pipeline(
        df_fold_train.drop(columns=["label"]),
        y_fold_train,
        test_df=df_fold_val,
        random_state=fold
    )
    
    acc = accuracy_score(y_fold_val, y_pred)
    bal = balanced_accuracy_score(y_fold_val, y_pred)
    mcc = matthews_corrcoef(y_fold_val, y_pred)
    
    cv_scores_acc.append(acc)
    cv_scores_bal.append(bal)
    cv_scores_mcc.append(mcc)
    all_y_true.extend(y_fold_val)
    all_y_pred.extend(y_pred)
    
    print(f"  → accuracy={acc:.4f}, balanced_acc={bal:.4f}, MCC={mcc:.4f}")
    fold += 1

print(f"\n" + "=" * 60)
print("RÉSULTATS CROSS-VALIDATION (5 folds)")
print("=" * 60)
print(f"  Accuracy:        {np.mean(cv_scores_acc):.4f} ± {np.std(cv_scores_acc):.4f}")
print(f"  Balanced Acc:    {np.mean(cv_scores_bal):.4f} ± {np.std(cv_scores_bal):.4f}")
print(f"  MCC:             {np.mean(cv_scores_mcc):.4f} ± {np.std(cv_scores_mcc):.4f}")
print(f"\n  Matrice de confusion (cumulée sur tous les folds) :")
print(confusion_matrix(all_y_true, all_y_pred))
print(f"\n  Rapport de classification (cumulé) :")
print(classification_report(all_y_true, all_y_pred, digits=4))

Train set: (1211, 2653)  ({'control': np.int64(892), 'II': np.int64(319)})
Test set (réservé pour évaluation finale): (303, 2653)  ({'control': np.int64(223), 'II': np.int64(80)})
[OK] Split effectué, le test est mis de côté jusqu'à la fin

VALIDATION CROISÉE 5-FOLD

  ── Fold 1 ──
    Composantes PCA: 2
  → accuracy=0.9959, balanced_acc=0.9972, MCC=0.9895

  ── Fold 2 ──
    Composantes PCA: 1
  → accuracy=0.9959, balanced_acc=0.9972, MCC=0.9894

  ── Fold 3 ──
    Composantes PCA: 2
  → accuracy=0.9835, balanced_acc=0.9688, MCC=0.9575

  ── Fold 4 ──
    Composantes PCA: 2
  → accuracy=0.9917, balanced_acc=0.9944, MCC=0.9792

  ── Fold 5 ──
    Composantes PCA: 2
  → accuracy=1.0000, balanced_acc=1.0000, MCC=1.0000

RÉSULTATS CROSS-VALIDATION (5 folds)
  Accuracy:        0.9934 ± 0.0056
  Balanced Acc:    0.9915 ± 0.0115
  MCC:             0.9831 ± 0.0144

  Matrice de confusion (cumulée sur tous les folds) :
[[315   4]
 [  4 888]]

  Rapport de classification (cumulé) :
            

In [13]:
# ── CELLULE 3 : Entraînement final sur TOUT le train + évaluation sur test réservé ──
from sklearn.metrics import accuracy_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix, classification_report
import warnings
warnings.filterwarnings("ignore")

print("=" * 60)
print("ENTRAÎNEMENT FINAL SUR TOUT LE TRAIN")
print("=" * 60)

# On applique le pipeline à tout l'ensemble d'entraînement
rf_final, le_final, y_pred_test, y_true_test = run_pipeline(
    df_train_all.drop(columns=["label"]),
    y_train_all,
    test_df=df_test_final,
    random_state=42
)

print(f"\nTrain: {df_train_all.shape}, Test: {df_test_final.shape}")
# print(f"RNAs dans le test utilisés par le modèle: {rf_final.n_features_in_}")

# Évaluation sur le test set réservé (jamais vu par la CV)
print("\n" + "=" * 60)
print("ÉVALUATION SUR LE TEST SET RÉSERVÉ")
print("=" * 60)
acc_test = accuracy_score(y_true_test, y_pred_test)
bal_test = balanced_accuracy_score(y_true_test, y_pred_test)
mcc_test = matthews_corrcoef(y_true_test, y_pred_test)

print(f"  Accuracy:        {acc_test:.4f}")
print(f"  Balanced Acc:    {bal_test:.4f}")
print(f"  MCC:             {mcc_test:.4f}")
print(f"\n  Matrice de confusion :")
cm = confusion_matrix(y_true_test, y_pred_test)
print(cm)
print(f"\n  Rapport de classification :")
print(classification_report(y_true_test, y_pred_test, digits=4))

# Comparaison CV vs Test final (si les scores sont éloignés → overfitting)
cv_mean_acc = np.mean(cv_scores_acc)
print(f"\n  Comparaison : CV accuracy moyen = {cv_mean_acc:.4f} vs Test accuracy = {acc_test:.4f}")
if abs(cv_mean_acc - acc_test) > 0.1:
    print("  ⚠️ Écart > 0.1 entre CV et test → possible sur-apprentissage")
else:
    print("  ✅ Écart modéré entre CV et test → modèle plus fiable")

ENTRAÎNEMENT FINAL SUR TOUT LE TRAIN
    Composantes PCA: 2

Train: (1211, 2653), Test: (303, 2653)

ÉVALUATION SUR LE TEST SET RÉSERVÉ
  Accuracy:        0.9868
  Balanced Acc:    0.9750
  MCC:             0.9661

  Matrice de confusion :
[[ 76   4]
 [  0 223]]

  Rapport de classification :
              precision    recall  f1-score   support

          II     1.0000    0.9500    0.9744        80
     control     0.9824    1.0000    0.9911       223

    accuracy                         0.9868       303
   macro avg     0.9912    0.9750    0.9827       303
weighted avg     0.9870    0.9868    0.9867       303


  Comparaison : CV accuracy moyen = 0.9934 vs Test accuracy = 0.9868
  ✅ Écart modéré entre CV et test → modèle plus fiable


In [15]:
# ── CELLULE 4 : Récapitulatif complet des résultats ──
import numpy as np
from sklearn.metrics import confusion_matrix

print("=" * 60)
print("RÉCAPITULATIF FINAL")
print("=" * 60)

print(f"\n📊 VALIDATION CROISÉE 5-FOLD (sur {df_train_all.shape[0]} échantillons train) :")
print(f"  Accuracy:        {np.mean(cv_scores_acc):.4f} ± {np.std(cv_scores_acc):.4f}")
print(f"  Balanced Acc:    {np.mean(cv_scores_bal):.4f} ± {np.std(cv_scores_bal):.4f}")
print(f"  MCC:             {np.mean(cv_scores_mcc):.4f} ± {np.std(cv_scores_mcc):.4f}")
print(f"  Scores par fold : {[f'{s:.4f}' for s in cv_scores_acc]}")

print(f"\n📊 TEST SET RÉSERVÉ ({df_test_final.shape[0]} échantillons) :")
print(f"  Accuracy:        {acc_test:.4f}")
print(f"  Balanced Acc:    {bal_test:.4f}")
print(f"  MCC:             {mcc_test:.4f}")

print(f"\n📊 MATRICE DE CONFUSION (test set) :")
cm = confusion_matrix(y_true_test, y_pred_test)
print(f"                Prédit")
print(f"               control    II")
print(f"  Réel control   {cm[0][0]:>4d}    {cm[0][1]:>4d}")
print(f"        II       {cm[1][0]:>4d}    {cm[1][1]:>4d}")

# Éviter division par zéro
tn, fp, fn, tp = cm.ravel()
sens = tn / (tn + fp) if (tn + fp) > 0 else 0
spec = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"\n  Sensibilité (control bien classés): {sens:.4f}")
print(f"  Spécificité (II bien classés):      {spec:.4f}")

# Comparaison CV vs Test
print(f"\n📊 DIAGNOSTIC SUR-APPRENTISSAGE :")
ecart = abs(np.mean(cv_scores_acc) - acc_test)
print(f"  Écart CV mean vs Test accuracy: {ecart:.4f}")
if ecart > 0.10:
    print("  ⚠️ Écart > 0.10 → risque de sur-apprentissage")
elif ecart > 0.05:
    print("  ⚠️ Écart entre 0.05 et 0.10 → léger sur-apprentissage possible")
else:
    print("  ✅ Écart ≤ 0.05 → modèle stable et généralisable")

if np.mean(cv_scores_acc) >= 0.99:
    print("  ⚠️ Scores CV ~1.0 → le modèle pourrait encore mémoriser, vérifier avec moins de features ou plus de régularisation")

print(f"\n[OK] Analyse terminée — les résultats de la CV 5-fold sont  plus fiables")
print(f"    que le simple split train/test inital qui donnait des résultats non fiables.")

RÉCAPITULATIF FINAL

📊 VALIDATION CROISÉE 5-FOLD (sur 1211 échantillons train) :
  Accuracy:        0.9934 ± 0.0056
  Balanced Acc:    0.9915 ± 0.0115
  MCC:             0.9831 ± 0.0144
  Scores par fold : ['0.9959', '0.9959', '0.9835', '0.9917', '1.0000']

📊 TEST SET RÉSERVÉ (303 échantillons) :
  Accuracy:        0.9868
  Balanced Acc:    0.9750
  MCC:             0.9661

📊 MATRICE DE CONFUSION (test set) :
                Prédit
               control    II
  Réel control     76       4
        II          0     223

  Sensibilité (control bien classés): 0.9500
  Spécificité (II bien classés):      1.0000

📊 DIAGNOSTIC SUR-APPRENTISSAGE :
  Écart CV mean vs Test accuracy: 0.0066
  ✅ Écart ≤ 0.05 → modèle stable et généralisable
  ⚠️ Scores CV ~1.0 → le modèle pourrait encore mémoriser, vérifier avec moins de features ou plus de régularisation

[OK] Analyse terminée — les résultats de la CV 5-fold sont  plus fiables
    que le simple split train/test inital qui donnait des résultats 